In [2]:
import pandas as pd
import numpy as np
from scipy.stats import multivariate_normal

In [7]:
# create the phase labels df for training from the cleaned survey data
survey_df = pd.read_csv("final_clean_survey_risk.csv")



In [8]:
df = survey_df.copy()
df['day'] = pd.to_datetime(df['day']).dt.normalize()

cycle_meta = []

for pid, g in df.groupby('name'):
    g = g.sort_values('day').reset_index(drop=True)

    # All menses start days = cycle boundaries
    m_starts = (
        g.loc[g['menzie_flag'] == 1, 'day']
        .sort_values()
        .tolist()
    )

    # Ovulation days (may be empty)
    o_days = (
        g.loc[g['ovulation_flag'] == 1, 'day']
        .sort_values()
        .tolist()
    )

    if len(m_starts) == 0:
        continue  # no cycles possible without menses

    for i, m_start in enumerate(m_starts):

        # ------------------------------------
        # DEFINE CYCLE END
        # ------------------------------------
        if i < len(m_starts) - 1:
            # Next menses defines the end of current cycle
            next_m_start = m_starts[i+1]
            cycle_end = next_m_start - pd.Timedelta(days=1)
            is_truncated = False
        else:
            # Last cycle → truncated (no future menses)
            cycle_end = g['day'].max()
            is_truncated = True

        # Window of this cycle
        cyc = g[(g['day'] >= m_start) & (g['day'] <= cycle_end)]

        # ------------------------------------
        # OVULATION ASSIGNMENT
        # ------------------------------------
        ovulations_in_cycle = (
            cyc.loc[cyc['ovulation_flag'] == 1, 'day']
            .sort_values()
            .tolist()
        )

        # A valid ovulation must occur strictly inside this cycle
        has_ovulation = (
            len(ovulations_in_cycle) == 1
        ) and (
            ovulations_in_cycle[0] > m_start
        ) and (
            ovulations_in_cycle[0] <= cycle_end
        )

        # ------------------------------------
        # COMPLETENESS CRITERION
        # ------------------------------------
        is_complete = (not is_truncated) and has_ovulation

        # Classification of incomplete types
        is_menses_only = (not is_complete) and (len(ovulations_in_cycle) == 0)
        is_ovulation_only = (not is_complete) and (len(ovulations_in_cycle) > 0)
        is_unlabeled = False  # you always have menses; no need for this flag

        cycle_meta.append({
            'name': pid,
            'cycle_id': i + 1,
            'cycle_start': m_start,
            'cycle_end': cycle_end,
            'is_truncated': is_truncated,
            'has_menses': True,
            'has_ovulation': has_ovulation,
            'is_complete': is_complete,
            'is_menses_only': is_menses_only,
            'is_ovulation_only': is_ovulation_only,
            'ovulation_day': ovulations_in_cycle[0] if has_ovulation else pd.NaT,
            'num_days': (cycle_end - m_start).days + 1
        })

cycle_table = pd.DataFrame(cycle_meta)
cycle_table


,name,cycle_id,cycle_start,cycle_end,is_truncated,has_menses,has_ovulation,is_complete,is_menses_only,is_ovulation_only,ovulation_day,num_days
0,10200,1,2024-09-26,2024-10-19,False,True,False,False,True,False,NaT,24
1,10200,2,2024-10-20,2024-11-12,False,True,False,False,True,False,NaT,24
2,10200,3,2024-11-13,2024-12-05,False,True,False,False,True,False,NaT,23
3,10200,4,2024-12-06,2024-12-05,False,True,False,False,True,False,NaT,0
4,10200,5,2024-12-06,2024-12-29,False,True,False,False,True,False,NaT,24
...,...,...,...,...,...,...,...,...,...,...,...,...
194,11011,4,2025-07-31,2025-08-28,False,True,False,False,True,False,NaT,29
195,11011,5,2025-08-29,2025-10-25,False,True,False,False,True,False,NaT,58
196,11011,6,2025-10-26,2025-11-24,False,True,True,True,False,False,2025-11-13,30
197,11011,7,2025-11-25,2025-11-29,True,True,False,False,True,False,NaT,5


In [9]:
print("\n================= CYCLE SUMMARY =================")

summary = {
    "Total cycles":            len(cycle_table),
    "Complete cycles":         cycle_table["is_complete"].sum(),
    "Menses-only cycles":      cycle_table["is_menses_only"].sum(),
    "Ovulation-only cycles":   cycle_table["is_ovulation_only"].sum(),
    "Truncated cycles":        cycle_table["is_truncated"].sum()
}

for k, v in summary.items():
    print(f"{k:25} {v}")

print("\n================= VALIDATION (column sums) =================")
print(cycle_table[[
    "is_complete",
    "is_menses_only",
    "is_ovulation_only",
    "is_truncated"
]].sum())



================= CYCLE SUMMARY =================
Total cycles              199
Complete cycles           58
Menses-only cycles        128
Ovulation-only cycles     13
Truncated cycles          23

================= VALIDATION (column sums) =================
is_complete           58
is_menses_only       128
is_ovulation_only     13
is_truncated          23
dtype: int64


In [10]:
# create the actual phase df

phase_rows_complete = []
phase_rows_partial = []

for pid, cyc in cycle_table.groupby('name'):
    # Subset participant survey rows
    g = survey_df[survey_df['name'] == pid].sort_values('day')

    for _, c in cyc.iterrows():
        cid = c['cycle_id']
        start = c['cycle_start']
        end = c['cycle_end']
        is_complete = c['is_complete']
        ovu = c['ovulation_day']
        
        # Restrict to this cycle window
        days = pd.date_range(start, end, freq='D')
        dfc = pd.DataFrame({'name': pid, 'day': days})
        dfc['cycle_id'] = cid
        
        # -------------------------------
        # COMPLETE CYCLES — FULL LABELING
        # -------------------------------
        if is_complete:
            dfc['phase'] = 'L'  # default luteal

            # Menses = 4 days
            m_end = start + pd.Timedelta(days=3)
            dfc.loc[dfc['day'].between(start, m_end), 'phase'] = 'M'

            # Ovulation day
            dfc.loc[dfc['day'] == ovu, 'phase'] = 'O'

            # Follicular = after menses until day before ovulation
            dfc.loc[
                (dfc['day'] > m_end) & (dfc['day'] < ovu),
                'phase'
            ] = 'F'

            # day_in_phase
            dfc['day_in_phase'] = dfc.groupby('phase').cumcount() + 1

            phase_rows_complete.append(dfc)

        # -------------------------------
        # PARTIAL CYCLES — ONLY M & O
        # -------------------------------
        else:
            dfp = dfc.copy()
            dfp['phase'] = np.nan

            # find actual menses days from survey_df
            m_days = pd.to_datetime(g.loc[g['menzie_flag'] == 1, 'day']).tolist()


            # label M (4 days)
            for m_start in m_days:
                if m_start >= start and m_start <= end:
                    m_end = m_start + pd.Timedelta(days=3)
                    dfp.loc[dfp['day'].between(m_start, m_end), 'phase'] = 'M'

            # label O days
            o_days = pd.to_datetime(g.loc[g['ovulation_flag'] == 1, 'day']).tolist()

            for o_day in o_days:
                if o_day >= start and o_day <= end:
                    dfp.loc[dfp['day'] == o_day, 'phase'] = 'O'

            dfp['day_in_phase'] = dfp.groupby('phase').cumcount() + 1

            phase_rows_partial.append(dfp)

# Combine
phase_daily_complete = pd.concat(phase_rows_complete).reset_index(drop=True)
phase_daily_partial  = pd.concat(phase_rows_partial).reset_index(drop=True)

phase_daily = pd.concat([phase_daily_complete, phase_daily_partial]).reset_index(drop=True)

phase_daily.head()


C:\Users\alina\AppData\Local\Temp\ipykernel_17464\869856038.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'M' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dfp.loc[dfp['day'].between(m_start, m_end), 'phase'] = 'M'
C:\Users\alina\AppData\Local\Temp\ipykernel_17464\869856038.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'M' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dfp.loc[dfp['day'].between(m_start, m_end), 'phase'] = 'M'
C:\Users\alina\AppData\Local\Temp\ipykernel_17464\869856038.py:61: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'M' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  dfp.loc[dfp[

,name,day,cycle_id,phase,day_in_phase
0,10200,2025-03-20,10,M,1.0
1,10200,2025-03-21,10,M,2.0
2,10200,2025-03-22,10,M,3.0
3,10200,2025-03-23,10,M,4.0
4,10200,2025-03-24,10,F,1.0


In [11]:
# count how many days each phase lasts for each participant cycle
cycle_phase_durations = (
    phase_daily_complete
        .groupby(['name','cycle_id','phase'])
        .size()
        .reset_index(name='duration')
)

cycle_phase_durations.head()

,name,cycle_id,phase,duration
0,10200,10,F,10
1,10200,10,L,9
2,10200,10,M,4
3,10200,10,O,1
4,10200,19,F,8


In [12]:
# Get distribution table
duration_dist = (
    cycle_phase_durations
        .groupby(['phase', 'duration'])
        .size()
        .unstack(fill_value=0)
)

duration_dist


duration,1,4,5,6,7,8,9,10,11,12,...,20,21,22,26,28,39,41,42,51,66
phase,,,,,,,,,,,,,,,,,,,,,
F,0,1,2,3,6,8,8,7,4,3,...,1,1,0,0,1,0,0,0,0,1
L,0,0,0,0,0,0,3,1,6,3,...,0,0,1,1,0,1,1,1,1,0
M,0,58,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
O,58,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [13]:
duration_probs = duration_dist.div(duration_dist.sum(axis=1), axis=0)
duration_probs


duration,1,4,5,6,7,8,9,10,11,12,...,20,21,22,26,28,39,41,42,51,66
phase,,,,,,,,,,,,,,,,,,,,,
F,0.0,0.017241,0.034483,0.051724,0.103448,0.137931,0.137931,0.120690,0.068966,0.051724,...,0.017241,0.017241,0.000000,0.000000,0.017241,0.000000,0.000000,0.000000,0.000000,0.017241
L,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.051724,0.017241,0.103448,0.051724,...,0.000000,0.000000,0.017241,0.017241,0.000000,0.017241,0.017241,0.017241,0.017241,0.000000
M,0.0,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
O,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [14]:
duration_dict = {
    phase: duration_probs.loc[phase].to_dict()
    for phase in duration_probs.index
}

duration_dict


{'F': {1: 0.0,
  4: 0.017241379310344827,
  5: 0.034482758620689655,
  6: 0.05172413793103448,
  7: 0.10344827586206896,
  8: 0.13793103448275862,
  9: 0.13793103448275862,
  10: 0.1206896551724138,
  11: 0.06896551724137931,
  12: 0.05172413793103448,
  13: 0.05172413793103448,
  14: 0.08620689655172414,
  15: 0.017241379310344827,
  16: 0.034482758620689655,
  17: 0.0,
  18: 0.017241379310344827,
  19: 0.0,
  20: 0.017241379310344827,
  21: 0.017241379310344827,
  22: 0.0,
  26: 0.0,
  28: 0.017241379310344827,
  39: 0.0,
  41: 0.0,
  42: 0.0,
  51: 0.0,
  66: 0.017241379310344827},
 'L': {1: 0.0,
  4: 0.0,
  5: 0.0,
  6: 0.0,
  7: 0.0,
  8: 0.0,
  9: 0.05172413793103448,
  10: 0.017241379310344827,
  11: 0.10344827586206896,
  12: 0.05172413793103448,
  13: 0.1724137931034483,
  14: 0.20689655172413793,
  15: 0.1724137931034483,
  16: 0.05172413793103448,
  17: 0.034482758620689655,
  18: 0.017241379310344827,
  19: 0.017241379310344827,
  20: 0.0,
  21: 0.0,
  22: 0.017241379310344

In [15]:
# split cycles into complete cycles for training, complete for testing, partial for testing

# Make a copy to avoid modifying original
cycle_table_split = cycle_table.copy()

# Extract complete cycle rows
complete_cycles = cycle_table_split[cycle_table_split['is_complete']].copy()

# Fix random seed for reproducibility
np.random.seed(42)

# Randomly select 13 complete cycles for held-out testing
heldout_idx = np.random.choice(
    complete_cycles.index, 
    size=13, 
    replace=False
)

# Mark them
cycle_table_split['set'] = 'train'     # default
cycle_table_split.loc[heldout_idx, 'set'] = 'test_complete'

# All incomplete (non-complete) cycles are test_partial
cycle_table_split.loc[~cycle_table_split['is_complete'], 'set'] = 'test_partial'

# Extract lists
train_complete_cycles = cycle_table_split[
    (cycle_table_split['is_complete']) &
    (cycle_table_split['set'] == 'train')
]

test_complete_cycles = cycle_table_split[
    (cycle_table_split['is_complete']) &
    (cycle_table_split['set'] == 'test_complete')
]


test_partial_cycles = cycle_table_split[
    cycle_table_split['set'] == 'test_partial'
]

cycle_table_split.head()


,name,cycle_id,cycle_start,cycle_end,is_truncated,has_menses,has_ovulation,is_complete,is_menses_only,is_ovulation_only,ovulation_day,num_days,set
0,10200,1,2024-09-26,2024-10-19,False,True,False,False,True,False,NaT,24,test_partial
1,10200,2,2024-10-20,2024-11-12,False,True,False,False,True,False,NaT,24,test_partial
2,10200,3,2024-11-13,2024-12-05,False,True,False,False,True,False,NaT,23,test_partial
3,10200,4,2024-12-06,2024-12-05,False,True,False,False,True,False,NaT,0,test_partial
4,10200,5,2024-12-06,2024-12-29,False,True,False,False,True,False,NaT,24,test_partial


In [16]:
print("===== FINAL SPLIT =====")
print("Training complete cycles:", len(train_complete_cycles))
print("Held-out complete cycles:", len(test_complete_cycles))
print("Test partial cycles:", len(test_partial_cycles))
print("Total cycles:", len(cycle_table_split))


===== FINAL SPLIT =====
Training complete cycles: 45
Held-out complete cycles: 13
Test partial cycles: 141
Total cycles: 199


In [17]:
# ---------------------------------------
# Build lookup sets (name, cycle_id) pairs
# ---------------------------------------

train_ids = {
    (row.name, row.cycle_id)
    for row in train_complete_cycles.itertuples(index=False)
}

test_complete_ids = {
    (row.name, row.cycle_id)
    for row in test_complete_cycles.itertuples(index=False)
}

test_partial_ids = {
    (row.name, row.cycle_id)
    for row in test_partial_cycles.itertuples(index=False)
}


# ---------------------------------------
# 1. TRAIN = complete cycles not held out
# ---------------------------------------

phase_train = phase_daily_complete[
    phase_daily_complete.apply(
        lambda r: (r['name'], r['cycle_id']) in train_ids,
        axis=1
    )
].reset_index(drop=True)


# ---------------------------------------
# 2. TEST (COMPLETE) = 13 held-out cycles
# ---------------------------------------

phase_test_complete = phase_daily_complete[
    phase_daily_complete.apply(
        lambda r: (r['name'], r['cycle_id']) in test_complete_ids,
        axis=1
    )
].reset_index(drop=True)


# ---------------------------------------
# 3. TEST (PARTIAL) = all incomplete cycles
# ---------------------------------------

phase_test_partial = phase_daily_partial[
    phase_daily_partial.apply(
        lambda r: (r['name'], r['cycle_id']) in test_partial_ids,
        axis=1
    )
].reset_index(drop=True)


# ---------------------------------------
# PRINT SUMMARY
# ---------------------------------------
print("===== PHASE DAILY SPLIT =====")
print("Train rows:          ", len(phase_train))
print("Test complete rows:  ", len(phase_test_complete))
print("Test partial rows:   ", len(phase_test_partial))

print("\nTrain cycles:        ", len(train_ids))
print("Test complete cycles:", len(test_complete_ids))
print("Test partial cycles: ", len(test_partial_ids))



===== PHASE DAILY SPLIT =====
Train rows:           1438
Test complete rows:   448
Test partial rows:    4469

Train cycles:         45
Test complete cycles: 13
Test partial cycles:  141


In [18]:
# Save to CSV
phase_train.to_csv("phase_daily_train.csv", index=False)
phase_test_complete.to_csv("phase_daily_test_complete.csv", index=False)
phase_test_partial.to_csv("phase_daily_test_partial.csv", index=False)

In [19]:
# now create the combined biometric and survey risk data
survey_risk = pd.read_csv("final_clean_survey_risk.csv", parse_dates=["day"])
survey_risk['day'] = survey_risk['day'].dt.normalize()
biom = pd.read_csv("smoothed_biometrics.csv", parse_dates=["day"])
biom['day'] = biom['day'].dt.normalize()

In [20]:
def merge_all(phase_df):
    out = (
        phase_df
        .merge(biom, on=['name','day'], how='left')
        .merge(
            survey_risk[['name','day','risk_obs','risk_latent_train','menzie_flag','ovulation_flag']],
            on=['name','day'], 
            how='left'
        )
    )
    return out


In [21]:
train_df = merge_all(phase_train)
test_complete_df = merge_all(phase_test_complete)
test_partial_df = merge_all(phase_test_partial)
train_df.head(10)

,name,day,cycle_id,phase,day_in_phase,hrv_smooth,breath_smooth,temp_smooth,risk_obs,risk_latent_train,menzie_flag,ovulation_flag
0,10200,2025-09-28,19,M,1,41.979200,14.338622,36.339086,1.0,1.0,1.0,0.0
1,10200,2025-09-29,19,M,2,42.226618,14.312827,36.043093,1.0,1.0,0.0,0.0
2,10200,2025-09-30,19,M,3,42.611438,14.268534,35.933423,1.0,1.0,0.0,0.0
3,10200,2025-10-01,19,M,4,42.900717,14.209328,36.223709,1.0,1.0,0.0,0.0
4,10200,2025-10-02,19,F,1,42.920641,14.137616,36.334053,1.0,1.0,0.0,0.0
5,10200,2025-10-03,19,F,2,42.603626,14.066811,36.207955,1.0,1.0,0.0,0.0
6,10200,2025-10-04,19,F,3,42.106293,14.018502,36.315473,1.0,1.0,0.0,0.0
7,10200,2025-10-05,19,F,4,41.546072,14.000583,36.424523,0.0,0.0,0.0,0.0
8,10200,2025-10-06,19,F,5,40.888559,13.996437,36.166227,0.0,0.0,0.0,0.0
9,10200,2025-10-07,19,F,6,40.246062,13.990000,36.030864,0.0,0.0,0.0,0.0


In [22]:
print("TRAIN shape:", train_df.shape)
print("TEST complete shape:", test_complete_df.shape)
print("TEST partial shape:", test_partial_df.shape)

print("Missing biometrics (train):")
print(train_df[['hrv_smooth','breath_smooth','temp_smooth']].isna().sum())

print("Missing risk or flags (train):")
print(train_df[['risk_obs','menzie_flag','ovulation_flag']].isna().sum())

# rows with missing risk_obs and menzie and ov flags were missing surveys

TRAIN shape: (1447, 12)
TEST complete shape: (448, 12)
TEST partial shape: (4523, 12)
Missing biometrics (train):
hrv_smooth       3
breath_smooth    3
temp_smooth      3
dtype: int64
Missing risk or flags (train):
risk_obs          63
menzie_flag       63
ovulation_flag    63
dtype: int64


In [23]:
# show the rows in train where risk_obs is NA
train_df[train_df['risk_obs'].isna()]

,name,day,cycle_id,phase,day_in_phase,hrv_smooth,breath_smooth,temp_smooth,risk_obs,risk_latent_train,menzie_flag,ovulation_flag
216,10215,2025-06-22,5,L,14,38.398374,14.093608,36.458905,NaN,NaN,NaN,NaN
217,10215,2025-06-23,5,L,15,39.636366,14.059009,36.437887,NaN,NaN,NaN,NaN
218,10215,2025-06-24,5,L,16,40.397272,14.017487,36.416499,NaN,NaN,NaN,NaN
219,10215,2025-06-25,5,L,17,41.023477,13.970751,36.395808,NaN,NaN,NaN,NaN
220,10215,2025-06-26,5,L,18,42.170999,13.921078,36.376843,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
1380,11007,2025-06-17,3,L,36,67.481546,16.553258,37.079961,NaN,NaN,NaN,NaN
1381,11007,2025-06-18,3,L,37,67.164169,16.574229,36.884988,NaN,NaN,NaN,NaN
1382,11007,2025-06-19,3,L,38,67.025863,16.629141,36.819992,NaN,NaN,NaN,NaN
1383,11007,2025-06-20,3,L,39,67.151658,16.674830,36.477513,NaN,NaN,NaN,NaN


In [24]:
# missing menzie and ovulation flags from train filled with zeros becasue these are missing survey coverage but still counted as complete cycles meaning that cycle had a menstruation and ovulation
event_cols = ['menzie_flag', 'ovulation_flag']

train_df[event_cols] = train_df[event_cols].fillna(0).astype(int)
test_complete_df[event_cols] = test_complete_df[event_cols].fillna(0).astype(int)





In [25]:
#save to csv
train_df.to_csv("final_pomdp_train.csv", index=False)
test_complete_df.to_csv("final_pomdp_test_complete.csv", index=False)
test_partial_df.to_csv("final_pomdp_test_partial.csv", index=False)

In [95]:
# read final dfs from csvs but drop col in all breath_smooth
train_df = pd.read_csv("final_pomdp_train.csv")
test_complete_df = pd.read_csv("final_pomdp_test_complete.csv")
test_partial_df = pd.read_csv("final_pomdp_test_partial.csv")
train_df = train_df.drop(columns=['breath_smooth'])
test_complete_df = test_complete_df.drop(columns=['breath_smooth'])
test_partial_df = test_partial_df.drop(columns=['breath_smooth'])

In [97]:
# ------------------------------------------------------------
# 1. Standardize HRV + Temp per participant (critical)
# ------------------------------------------------------------
def compute_train_stats(train_df):
    stats = {}
    for pid, g in train_df.groupby("name"):
        mu_hrv  = g["hrv_smooth"].mean()
        sd_hrv  = g["hrv_smooth"].std()
        mu_temp = g["temp_smooth"].mean()
        sd_temp = g["temp_smooth"].std()

        # Handle degenerate cases
        if not np.isfinite(sd_hrv)  or sd_hrv  < 1e-8: sd_hrv  = 1.0
        if not np.isfinite(sd_temp) or sd_temp < 1e-8: sd_temp = 1.0

        stats[pid] = {
            "mu_hrv": mu_hrv, "sd_hrv": sd_hrv,
            "mu_temp": mu_temp, "sd_temp": sd_temp
        }
    return stats

def apply_standardization(df, train_stats):
    df = df.copy()
    for pid, g in df.groupby("name"):

        if pid in train_stats:
            ts = train_stats[pid]
            mu_hrv, sd_hrv   = ts["mu_hrv"], ts["sd_hrv"]
            mu_temp, sd_temp = ts["mu_temp"], ts["sd_temp"]
        else:
            # fallback for unseen participants
            mu_hrv  = df["hrv_smooth"].mean()
            sd_hrv  = df["hrv_smooth"].std()
            mu_temp = df["temp_smooth"].mean()
            sd_temp = df["temp_smooth"].std()

            if not np.isfinite(sd_hrv)  or sd_hrv  < 1e-8: sd_hrv  = 1.0
            if not np.isfinite(sd_temp) or sd_temp < 1e-8: sd_temp = 1.0

        df.loc[g.index, "hrv_smooth"]  = (g["hrv_smooth"]  - mu_hrv)  / sd_hrv
        df.loc[g.index, "temp_smooth"] = (g["temp_smooth"] - mu_temp) / sd_temp

    return df

def preprocess_test_df(df, train_stats):
    df = df.copy().sort_values(["name", "day"])

    df["true_onset"] = (df["menzie_flag"] == 1).astype(int)

    df = apply_standardization(df, train_stats)

    df["biometrics"] = df.apply(
        lambda row: np.array([row["hrv_smooth"], row["temp_smooth"]]),
        axis=1
    )

    return df

# apply standardization to the TRAIN SET ONLY
# Compute stats ONCE from training data
train_stats = compute_train_stats(train_df)

# Standardize training using training stats
train_std = apply_standardization(train_df, train_stats)

# ------------------------------------------------------------
# 2. Build HSMM Gaussian emissions from standardized TRAIN data
# ------------------------------------------------------------
em_train = train_std.dropna(subset=['hrv_smooth', 'temp_smooth']).copy()
phases = ['M', 'F', 'O', 'L']
emission_params = {}

for ph in phases:
    sub = em_train[em_train['phase'] == ph][['hrv_smooth', 'temp_smooth']]

    if len(sub) == 0:
        print(f"WARNING: no data for phase {ph}")
        continue

    mean_vec = sub.mean().values       # shape (2,)
    cov_mat  = sub.cov().values        # shape (2,2)

    # numerical safety
    cov_mat += np.eye(2) * 1e-6

    emission_params[ph] = {
        "mean": mean_vec,
        "cov": cov_mat
    }


print("\n=== HSMM Gaussian Emission Parameters (STANDARDIZED) ===")
for k,v in emission_params.items():
    print(f"\nPhase {k}:")
    print("Mean:", v['mean'])
    print("Covariance:\n", v['cov'])



=== HSMM Gaussian Emission Parameters (STANDARDIZED) ===

Phase M:
Mean: [0.24381928 0.22367503]
Covariance:
 [[ 0.85532111 -0.11773655]
 [-0.11773655  0.82024245]]

Phase F:
Mean: [ 0.37382581 -0.35427147]
Covariance:
 [[0.87098363 0.11718846]
 [0.11718846 0.76796318]]

Phase O:
Mean: [ 0.1137365  -0.53435435]
Covariance:
 [[0.72929932 0.19205265]
 [0.19205265 0.59437463]]

Phase L:
Mean: [-0.33884254  0.23386326]
Covariance:
 [[ 0.89363419 -0.09565935]
 [-0.09565935  1.0405141 ]]


In [98]:
phase_order = ['M', 'F', 'O', 'L']
next_phase = {
    'M': 'F',
    'F': 'O',
    'O': 'L',
    'L': 'M'
}

hsmm_params = {
    "emissions": emission_params,
    "durations": duration_dict,
    "transitions": next_phase,
    "phases": phase_order
}

print("\nHSMM parameters ready.")


HSMM parameters ready.


In [99]:
# now build the risk transition and observation model

# -----------------------------------
# 1. Risk transition model P(R_{t+1} | R_t, Phase_t)
# -----------------------------------
train_df['day'] = pd.to_datetime(train_df['day'], errors='coerce')

# Use only rows where risk_latent_train is observed
rt = train_df[['name','day','phase','risk_latent_train']].dropna(subset=['risk_latent_train']).copy()
rt['risk_latent_train'] = rt['risk_latent_train'].astype(int)

# Sort by participant + day
rt = rt.sort_values(['name','day'])

# Build next-day risk and phase
rt['risk_next'] = rt.groupby('name')['risk_latent_train'].shift(-1)
rt['phase_next'] = rt.groupby('name')['phase'].shift(-1)
rt['day_next'] = rt.groupby('name')['day'].shift(-1)

# Keep only true day-to-day transitions
rt = rt[(rt['day_next'] == rt['day'] + pd.Timedelta(days=1))]
rt = rt.dropna(subset=['risk_next', 'phase_next'])
rt['risk_next'] = rt['risk_next'].astype(int)

phases = hsmm_params['phases']  # ['M','F','O','L']
risk_states = [0, 1]

# Laplace smoothing: start with 1 count in each cell
risk_trans_counts = {ph: np.ones((2, 2)) for ph in phases}
global_counts = np.ones((2, 2))

for _, row in rt.iterrows():
    ph = row['phase']
    r  = int(row['risk_latent_train'])
    r2 = int(row['risk_next'])
    if ph in risk_trans_counts:
        risk_trans_counts[ph][r, r2] += 1
    global_counts[r, r2] += 1

# Normalize rows to get probabilities
def row_normalize(mat):
    mat = mat.astype(float)
    row_sums = mat.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return mat / row_sums

risk_transition = {ph: row_normalize(risk_trans_counts[ph]) for ph in phases}
risk_transition_global = row_normalize(global_counts)

print("Risk transition matrices per phase (rows = R_t, cols = R_{t+1}):")
for ph in phases:
    print(f"\nPhase {ph}:\n", risk_transition[ph])

print("\nGlobal risk transition matrix:\n", risk_transition_global)

Risk transition matrices per phase (rows = R_t, cols = R_{t+1}):

Phase M:
 [[0.98181818 0.01818182]
 [0.12295082 0.87704918]]

Phase F:
 [[0.98853868 0.01146132]
 [0.10294118 0.89705882]]

Phase O:
 [[0.97297297 0.02702703]
 [0.08333333 0.91666667]]

Phase L:
 [[0.93095238 0.06904762]
 [0.02222222 0.97777778]]

Global risk transition matrix:
 [[0.9625731  0.0374269 ]
 [0.06543967 0.93456033]]


In [100]:

# ============================================================
# 1B. CARE-modified risk transition model
# ============================================================

boost = 0.25   # tune this

risk_transition_care = {}

for ph in phases:
    base = risk_transition[ph].copy()
    care = base.copy()

    # Only modify transitions out of high-risk (state 1)
    increase = min(base[1,0] + boost, 1.0)
    care[1,0] = increase
    care[1,1] = 1.0 - increase

    # low-risk (state 0) unchanged
    risk_transition_care[ph] = row_normalize(care)

print("\nCARE risk transition matrices:")
for ph in phases:
    print(f"\nPhase {ph}:\n", risk_transition_care[ph])



CARE risk transition matrices:

Phase M:
 [[0.98181818 0.01818182]
 [0.37295082 0.62704918]]

Phase F:
 [[0.98853868 0.01146132]
 [0.35294118 0.64705882]]

Phase O:
 [[0.97297297 0.02702703]
 [0.33333333 0.66666667]]

Phase L:
 [[0.93095238 0.06904762]
 [0.27222222 0.72777778]]


In [101]:
# ============================================================
# ========== 3. NEW: Semi-Markov Phase Transitions ============
# ============================================================

duration_params = hsmm_params["durations"]     # Your HSMM durations
phase_index = {ph:i for i,ph in enumerate(phases)}
nP = len(phases)

phase_mean_duration = {}
phase_p_self = {}
phase_p_change = {}

for ph in phases:
    dist = duration_params[ph]   # {duration: prob}
    mean_dur = sum(d * p for d, p in dist.items())
    phase_mean_duration[ph] = mean_dur
    
    p_self = max((mean_dur - 1) / mean_dur, 0)
    p_change = 1 - p_self

    phase_p_self[ph] = p_self
    phase_p_change[ph] = p_change

# Build Phase Transition Matrix (4×4)
Phase_T = np.zeros((nP, nP))

# M → M/F
Phase_T[phase_index["M"], phase_index["M"]] = phase_p_self["M"]
Phase_T[phase_index["M"], phase_index["F"]] = phase_p_change["M"]

# F → F/O
Phase_T[phase_index["F"], phase_index["F"]] = phase_p_self["F"]
Phase_T[phase_index["F"], phase_index["O"]] = phase_p_change["F"]

# O → L only (duration=1 → p_self=0)
Phase_T[phase_index["O"], phase_index["O"]] = phase_p_self["O"]
Phase_T[phase_index["O"], phase_index["L"]] = phase_p_change["O"]

# L → L/M
Phase_T[phase_index["L"], phase_index["L"]] = phase_p_self["L"]
Phase_T[phase_index["L"], phase_index["M"]] = phase_p_change["L"]

print("Corrected Phase Transition Matrix:")
print(pd.DataFrame(Phase_T, index=phases, columns=phases))


Corrected Phase Transition Matrix:
          M         F         O         L
M  0.750000  0.250000  0.000000  0.000000
F  0.000000  0.913303  0.086697  0.000000
O  0.000000  0.000000  0.000000  1.000000
L  0.062567  0.000000  0.000000  0.937433


In [102]:
# ============================================================
# ========== 4. NEW: Build Correct Joint Transition ===========
# ============================================================

state_list = []
state_index = {}
idx = 0
for ph in phases:
    for r in risk_states:
        state_list.append((ph, r))
        state_index[(ph, r)] = idx
        idx += 1

def build_joint_matrix(Phase_T, risk_T_dict):
    """
    Phase_T:    4×4 matrix of P(ph'|ph)
    risk_T_dict: dict phase -> 2×2 risk transitions
    """
    nS = len(state_list)
    T = np.zeros((nS, nS))

    for ph in phases:
        ph_i = phase_index[ph]
        for r in risk_states:
            s = state_index[(ph, r)]

            for ph2 in phases:
                ph_j = phase_index[ph2]
                ph_prob = Phase_T[ph_i, ph_j]   # phase transition probability

                for r2 in risk_states:
                    r_prob = risk_T_dict[ph][r, r2]

                    s2 = state_index[(ph2, r2)]
                    T[s, s2] = ph_prob * r_prob

    return T

# Transitions per action
T_wait  = build_joint_matrix(Phase_T, risk_transition)
T_query = T_wait.copy()
T_alert = T_wait.copy()
T_care  = build_joint_matrix(Phase_T, risk_transition_care)

print("\nJoint Transition Matrix (WAIT):")
print(pd.DataFrame(T_wait))


Joint Transition Matrix (WAIT):
          0         1         2         3         4         5         6  \
0  0.736364  0.013636  0.245455  0.004545  0.000000  0.000000  0.000000   
1  0.092213  0.657787  0.030738  0.219262  0.000000  0.000000  0.000000   
2  0.000000  0.000000  0.902836  0.010468  0.085703  0.000994  0.000000   
3  0.000000  0.000000  0.094017  0.819287  0.008925  0.077772  0.000000   
4  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.972973   
5  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.083333   
6  0.058247  0.004320  0.000000  0.000000  0.000000  0.000000  0.872705   
7  0.001390  0.061177  0.000000  0.000000  0.000000  0.000000  0.020832   

          7  
0  0.000000  
1  0.000000  
2  0.000000  
3  0.000000  
4  0.027027  
5  0.916667  
6  0.064727  
7  0.916601  


In [103]:
# -----------------------------------
# 2. Risk observation model P(risk_obs | Risk_latent)
# -----------------------------------

# Use only rows where both latent risk and observed risk are present
obs_df = train_df.dropna(subset=['risk_latent_train', 'risk_obs']).copy()
obs_df['risk_latent_train'] = obs_df['risk_latent_train'].astype(int)
obs_df['risk_obs'] = obs_df['risk_obs'].astype(int)

risk_obs_emission = {}

for r in risk_states:
    sub = obs_df[obs_df['risk_latent_train'] == r]
    if len(sub) == 0:
        p = 0.5  # fallback if no data
    else:
        p = sub['risk_obs'].mean()
    risk_obs_emission[r] = p

print("\nRisk observation model P(risk_obs=1 | R):")
for r, p in risk_obs_emission.items():
    print(f"R={r}: p={p:.3f}")



Risk observation model P(risk_obs=1 | R):
R=0: p=0.000
R=1: p=1.000


In [104]:

# -----------------------------------
# 4. Observation model pieces
#    Biometrics: already Gaussian given Phase (hsmm_params['emissions'])
#    Symptoms: Bernoulli given Risk (risk_obs_emission)
# -----------------------------------

# Biometrics emission: X | (Phase=ph, Risk=r) ~ N(mean_ph, cov_ph)
# (independent of risk; we just reuse emission_params for each risk level)

biometric_emissions = emission_params # {phase: {'mean':..., 'cov':...}}

# Symptom emission: Y | (Phase, Risk=r) uses only R
# P(Y=1 | (ph, r)) = risk_obs_emission[r]
symptom_emission = {}
for ph in phases:
    for r in risk_states:
        symptom_emission[(ph, r)] = risk_obs_emission[r]

print("\nSymptom observation model P(Y=1 | Phase, Risk):")
for st, p in symptom_emission.items():
    print(f"{st}: p={p:.3f}")



Symptom observation model P(Y=1 | Phase, Risk):
('M', 0): p=0.000
('M', 1): p=1.000
('F', 0): p=0.000
('F', 1): p=1.000
('O', 0): p=0.000
('O', 1): p=1.000
('L', 0): p=0.000
('L', 1): p=1.000


In [105]:
# ============================================================
# 5. Rewards
# ============================================================

# ALERT reward depends on timing error Δ computed externally
def alert_reward(delta):
    d = abs(delta)
    if d <= 1:
        return 10
    if 2 <= d <= 3:
        return 3
    if delta < -3:
        return -5
    if delta > 1:
        return -10
    return 0

reward_table = {
    "WAIT":  0,
    "QUERY": -1,
    "CARE":  -1,    # tunable
    "ALERT": alert_reward
}



In [106]:
# -----------------------------------
# 5. Pack everything into a POMDP structure
# -----------------------------------

pomdp_model = {
    "states": state_list,
    "state_index": state_index,

    "T": {
        "WAIT":  T_wait,
        "QUERY": T_query,
        "CARE":  T_care,
        "ALERT": T_alert
    },

    "emissions": {
        "biometric": emission_params,
        "symptom": symptom_emission
    },

    "risk_transition_baseline": risk_transition,
    "risk_transition_care": risk_transition_care,

    "risk_obs_emission": risk_obs_emission,

    "reward": reward_table
}

print("\nPOMDP core model (with CARE action) is ready.")


POMDP core model (with CARE action) is ready.


In [107]:
# ============================================================
# Preprocessing: convert raw test_df into per-day POMDP inputs
# ============================================================

def preprocess_test_df(df, train_stats):
    """
    Convert raw test dataframe for POMDP evaluation:
    - Sort by participant + day
    - Add true_onset flag
    - Standardize HRV using TRAIN μ,σ (per participant)
    - Build biometrics vector for emission model
    """
    df = df.copy().sort_values(["name", "day"])

    # ----- true onset label -----
    df["true_onset"] = (df["menzie_flag"] == 1).astype(int)

    # ----- Standardize using training stats -----
    df = apply_standardization(df, train_stats)

    # ----- Build biometrics vector -----
    df["biometrics"] = df["hrv_smooth"].apply(lambda x: np.array([x]))

    return df


In [108]:


# ============================================================
# Belief update components
# ============================================================

def predict_belief(b, T):
    return b @ T

def obs_loglikelihood(obs, state, pomdp_model):
    ph, r = state

    # ----- biometrics log-likelihood -----
    mean = pomdp_model["emissions"]["biometric"][ph]["mean"]
    cov  = pomdp_model["emissions"]["biometric"][ph]["cov"]

    # CRITICAL: ensure obs["biometrics"] is a 1D numpy array (not wrapped in extra brackets)
    x = np.atleast_1d(obs["biometrics"]).flatten()
    mean = np.atleast_1d(mean).flatten()
    
    # Ensure cov is 2D
    cov = np.atleast_2d(cov)
    
    # Add small regularization to avoid singular matrix
    cov_regularized = cov + np.eye(cov.shape[0]) * 1e-8
    # # Reshape x to be 2D, explicitly (1 observation, 1 feature)
    # x_reshaped = x.reshape(1, -1)
    try:
        ll_biom = multivariate_normal.logpdf(
            x,
            mean=mean,
            cov=cov_regularized,
            allow_singular=True
        )
    except Exception as e:
        print(f"ERROR in logpdf: x shape {x.shape}, mean shape {mean.shape}, cov shape {cov.shape}")
        print(f"x={x}, mean={mean}, cov={cov}")
        ll_biom = -np.inf

    # ----- symptom log-likelihood -----
    y = obs["symptom"]

    if y is None:
        ll_sym = 0.0  # missing → uninformative
    else:
        p = pomdp_model["emissions"]["symptom"][(ph, r)]
        p = np.clip(p, 1e-6, 1 - 1e-6)  # avoid log(0)
        ll_sym = np.log(p) if y == 1 else np.log(1 - p)

    return ll_biom + ll_sym

def update_belief(b_pred, obs, pomdp_model):

    states = pomdp_model["states"]
    S = len(states)

    # ----- log prior -----
    log_b_pred = np.log(b_pred + 1e-20)

    # ----- log-likelihoods -----
    log_like = np.zeros(S)
    for i, s in enumerate(states):
        log_like[i] = obs_loglikelihood(obs, s, pomdp_model)

    # ----- unnormalized log posterior -----
    log_post = log_b_pred + log_like

    # ----- stable softmax normalization -----
    max_log = np.max(log_post)
    post = np.exp(log_post - max_log)
    post /= post.sum()

    return post


In [143]:
# ============================================================
# Threshold policy
# ============================================================

def threshold_policy(b, pomdp_model, tau_risk, tau_onset, entropy_cut):
    # marginal risk prob
    p_risk = b[[i for i,(ph,r) in enumerate(pomdp_model["states"]) if r == 1]].sum()

    # marginal late luteal prob (onset soon)
    luteal_indices = [i for i, (ph, r) in enumerate(pomdp_model["states"]) if ph == "L"]
    p_late = b[luteal_indices].sum()

    if p_risk > tau_risk:
        return "CARE"
    if p_late > tau_onset:
        return "ALERT"

    # entropy as uncertainty measure
    entropy = -np.sum(b * np.log(b + 1e-12))
    if entropy > entropy_cut:
        return "QUERY"

    return "WAIT"


# ============================================================
# Run threshold policy for one cycle
# ============================================================

def run_cycle_threshold(cycle_df, pomdp_model, tau_risk, tau_onset, entropy_cut):

    S = len(pomdp_model["states"])
    # Initialize belief from the first day's phase
    first_phase = cycle_df["phase"].iloc[0]

    b = np.zeros(S)
    b[state_index[(first_phase, 0)]] = 0.5
    b[state_index[(first_phase, 1)]] = 0.5

    alerted_day = None
    rewards = 0
    query_count = 0

    # true onset day for cycle
    true_onset_days = cycle_df.loc[cycle_df["true_onset"] == 1, "day"]
    if len(true_onset_days) == 0:
        return None  # skip cycles with no onset label
    true_onset_day = pd.to_datetime(true_onset_days.iloc[0])

    for _, row in cycle_df.iterrows():

        # ----- choose action -----
        a = threshold_policy(b, pomdp_model, tau_risk, tau_onset, entropy_cut)

        if a == "QUERY":
            query_count += 1

        # ----- reward -----
        if a in ["WAIT", "QUERY", "CARE"]:
            rewards += pomdp_model["reward"][a]

        if a == "ALERT" and alerted_day is None:
            delta = (row["day"] - true_onset_day).days
            rewards += pomdp_model["reward"]["ALERT"](delta)
            alerted_day = row["day"]

        # ----- belief prediction -----
        T = pomdp_model["T"][a]
        b_pred = predict_belief(b, T)

        # ----- belief update using today's obs -----
        risk_obs_val = row["risk_obs"]
        if pd.isna(risk_obs_val):
            # missing symptom → uninformative likelihood
            symptom = None
        else:
            symptom = int(risk_obs_val)
        obs = {
            "biometrics": row["biometrics"],
            "symptom": symptom
        }
        b = update_belief(b_pred, obs, pomdp_model)

    # end-of-cycle outcome
    if alerted_day is None:
        delta = None
        rewards += -15  # penalty for no alert
    else:
        delta = (alerted_day - true_onset_day).days

    return {
        "cycle_id": cycle_df["cycle_id"].iloc[0],
        "alert_day": alerted_day,
        "delta": delta,
        "queries": query_count,
        "reward": rewards
    }


In [144]:
# ============================================================
# Evaluate over a full dataset
# ============================================================

def evaluate_threshold(test_df_prep, pomdp_model, tau_risk, tau_onset, entropy_cut):

    results = []

    for cycle_id, cycle_df in test_df_prep.groupby("cycle_id"):
        out = run_cycle_threshold(cycle_df, pomdp_model, tau_risk, tau_onset, entropy_cut)
        if out is not None:
            results.append(out)

    df = pd.DataFrame(results)
    if len(df) == 0:
        return None

    # ----- Metrics -----
    alerted = df[df["delta"].notna()]
    timing_errors = alerted["delta"].abs()

    timely_alert_rate = (timing_errors <= 1).mean()
    false_alarm_rate = (alerted["delta"] <= -3).mean()
    miss_late_rate = df["delta"].isna().mean() + (alerted["delta"] > 1).mean()

    avg_queries = df["queries"].mean()
    avg_reward = df["reward"].mean()

    return {
        "timely_alert_rate": timely_alert_rate,
        "false_alarm_rate": false_alarm_rate,
        "miss_late_rate": miss_late_rate,
        "avg_queries": avg_queries,
        "avg_reward": avg_reward,
        "timing_error_distribution": timing_errors.describe(),
        "raw": df
    }



In [ ]:
# ============================================================
# Run evaluation on BOTH complete + partial preprocessed sets
# ============================================================

# Preprocess raw test sets
test_complete_df_prep = preprocess_test_df(test_complete_df, train_stats)
test_partial_df_prep  = preprocess_test_df(test_partial_df,  train_stats)

# Datetime formatting
test_complete_df_prep["day"] = pd.to_datetime(test_complete_df_prep["day"])
test_partial_df_prep["day"]  = pd.to_datetime(test_partial_df_prep["day"])

# Evaluate on *preprocessed* sets
tau_risk = 0.4
tau_onset = 0.5
entropy_cut = 1
results_complete = evaluate_threshold(test_complete_df_prep, pomdp_model, tau_risk, tau_onset, entropy_cut)
results_partial  = evaluate_threshold(test_partial_df_prep,  pomdp_model, tau_risk, tau_onset, entropy_cut)

print("\n===== THRESHOLD POLICY RESULTS: COMPLETE TEST SET =====")
display(results_complete)

print("\n===== THRESHOLD POLICY RESULTS: PARTIAL TEST SET =====")
display(results_partial)



===== THRESHOLD POLICY RESULTS: COMPLETE TEST SET =====
{'timely_alert_rate': 0.0, 'false_alarm_rate': 0.25, 'miss_late_rate': 1.1785714285714286, 'avg_queries': 8.857142857142858, 'avg_reward': -48.57142857142857, 'timing_error_distribution': count      4.00000
mean      63.00000
std       39.04698
min       18.00000
25%       39.00000
50%       63.50000
75%       87.50000
max      107.00000
Name: delta, dtype: float64, 'raw':    cycle_id  alert_day  delta  queries  reward
0         1        NaT    NaN       18     -60
1         3        NaT    NaN        5     -36
2         6 2025-08-24  107.0        5     -66
3         8 2025-07-14   81.0       19     -74
4        10        NaT    NaN        0     -39
5        11 2025-06-09  -46.0       11     -40
6        13 2025-08-08   18.0        4     -25}

===== THRESHOLD POLICY RESULTS: PARTIAL TEST SET =====
{'timely_alert_rate': 0.0, 'false_alarm_rate': 0.0625, 'miss_late_rate': 1.1755952380952381, 'avg_queries': 46.38095238095238, 'avg_re

In [146]:
#grid search
tau_risk_grid   = [0.3, 0.35, 0.4, 0.45, 0.5]
tau_onset_grid  = [0.15, 0.2, 0.25, 0.3]
entropy_grid    = [1.0, 1.1, 1.2, 1.3, 1.4]

search_results = []

print("Running grid search...\n")

for tau_risk in tau_risk_grid:
    for tau_onset in tau_onset_grid:
        for entropy_cut in entropy_grid:

            metrics = evaluate_threshold(
                test_complete_df_prep,
                pomdp_model,
                tau_risk,
                tau_onset,
                entropy_cut
            )

            if metrics is None:
                continue

            search_results.append({
                "tau_risk": tau_risk,
                "tau_onset": tau_onset,
                "entropy_cut": entropy_cut,
                **metrics
            })

search_results_df = pd.DataFrame(search_results)

print(f"Total combinations evaluated: {len(search_results_df)}")
display(search_results_df.head())


# ============================================================
# Select Best Parameter Settings
# (1) Filter = false alarms < 20%
# (2) Maximize timely alert rate
# ============================================================

# Safety: ensure required columns exist
required_cols = ["false_alarm_rate", "timely_alert_rate"]
missing_cols = [c for c in required_cols if c not in search_results_df.columns]
if missing_cols:
    raise ValueError(f"Missing columns in results: {missing_cols}")

filtered = search_results_df[search_results_df["false_alarm_rate"] < 0.20]

if len(filtered) == 0:
    print("\n⚠️ No parameter sets satisfy false-alarm < 20%. Showing top performers instead.\n")
    filtered = search_results_df.sort_values("false_alarm_rate").head(10)

# Best by timely alert rate
best_timely = filtered.loc[filtered["timely_alert_rate"].idxmax()]

print("\n================ BEST PARAMETER SET ================\n")
print(best_timely)

# ============================================================
# (Optional) Sort alternatives by tradeoff
# ============================================================

sorted_candidates = filtered.sort_values(
    ["timely_alert_rate", "avg_reward"],
    ascending=[False, False]
)

print("\nTop 10 candidates by timely alert rate and reward:\n")
display(sorted_candidates.head(10))



Running grid search...

Total combinations evaluated: 100
Total combinations evaluated: 100


,tau_risk,tau_onset,entropy_cut,timely_alert_rate,false_alarm_rate,miss_late_rate,avg_queries,avg_reward,timing_error_distribution,raw
0,0.3,0.15,1.0,0.0,0.0,1.142857,3.714286,-42.714286,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
1,0.3,0.15,1.1,0.0,0.0,1.142857,0.000000,-39.000000,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
2,0.3,0.15,1.2,0.0,0.0,1.142857,0.000000,-39.000000,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
3,0.3,0.15,1.3,0.0,0.0,1.142857,0.000000,-39.000000,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
4,0.3,0.15,1.4,0.0,0.0,1.142857,0.000000,-39.000000,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...



================ BEST PARAMETER SET ================

tau_risk                                                                   0.3
tau_onset                                                                 0.15
entropy_cut                                                                1.0
timely_alert_rate                                                          0.0
false_alarm_rate                                                           0.0
miss_late_rate                                                        1.142857
avg_queries                                                           3.714286
avg_reward                                                          -42.714286
timing_error_distribution    count     6.000000
mean     34.833333
std     ...
raw                             cycle_id  alert_day  delta  queries  reward...
Name: 0, dtype: object

Top 10 candidates by timely alert rate and reward:



,tau_risk,tau_onset,entropy_cut,timely_alert_rate,false_alarm_rate,miss_late_rate,avg_queries,avg_reward,timing_error_distribution,raw
81,0.5,0.15,1.1,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
82,0.5,0.15,1.2,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
83,0.5,0.15,1.3,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
84,0.5,0.15,1.4,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 34.833333 std ...,cycle_id alert_day delta queries reward...
87,0.5,0.20,1.2,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 36.833333 std ...,cycle_id alert_day delta queries reward...
88,0.5,0.20,1.3,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 36.833333 std ...,cycle_id alert_day delta queries reward...
89,0.5,0.20,1.4,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 36.833333 std ...,cycle_id alert_day delta queries reward...
92,0.5,0.25,1.2,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 37.500000 std ...,cycle_id alert_day delta queries reward...
93,0.5,0.25,1.3,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 37.500000 std ...,cycle_id alert_day delta queries reward...
94,0.5,0.25,1.4,0.0,0.0,1.142857,0.0,-37.857143,count 6.000000 mean 37.500000 std ...,cycle_id alert_day delta queries reward...


In [124]:
# =============================================================
# PBVI Step 1: Collect Belief Points
# =============================================================

def initial_belief_from_cycle(cycle_df):
    """Your initial belief assumption: (phase0 × risk uniform)."""
    S = len(state_list)
    first_phase = cycle_df["phase"].iloc[0]
    b = np.zeros(S)
    b[state_index[(first_phase,0)]] = 0.5
    b[state_index[(first_phase,1)]] = 0.5
    return b

belief_points = []

def collect_beliefs(test_df_prep):
    all_beliefs = []
    
    for _, cycle in test_df_prep.groupby("cycle_id"):
        b = initial_belief_from_cycle(cycle)
        all_beliefs.append(b.copy())
        
        for _, row in cycle.iterrows():
            obs = {
                "biometrics": row["biometrics"],
                "symptom": None if pd.isna(row["risk_obs"]) else int(row["risk_obs"])
            }
            
            # Predict under WAIT
            T = pomdp_model["T"]["WAIT"]
            b_pred = predict_belief(b, T)
            
            # Update belief
            b = update_belief(b_pred, obs, pomdp_model)
            all_beliefs.append(b.copy())
            
    return all_beliefs

belief_points = collect_beliefs(test_complete_df_prep)
print(f"Collected {len(belief_points)} belief points.")


Collected 455 belief points.


In [125]:
# =============================================================
# PBVI Step 2: Sample transitions + observations
# =============================================================

def sample_transition_and_observation(s, a, pomdp_model):
    """Sample s' and observation o = (biometrics, symptom)."""
    
    # Sample next state
    T = pomdp_model["T"][a]
    s_prime = np.random.choice(len(state_list), p=T[s])
    ph_prime, r_prime = state_list[s_prime]
    
    # Sample biometric observation ~ Gaussian
    mean = pomdp_model["emissions"]["biometric"][ph_prime]["mean"]
    cov  = pomdp_model["emissions"]["biometric"][ph_prime]["cov"]
    biom = np.random.multivariate_normal(mean, cov).flatten()
    
    # Sample symptom ~ Bernoulli
    p_sym = pomdp_model["emissions"]["symptom"][(ph_prime, r_prime)]
    symptom = 1 if np.random.rand() < p_sym else 0
    
    obs = {"biometrics": biom, "symptom": symptom}
    return s_prime, obs


In [126]:
# =============================================================
# PBVI Step 3: Backup operator for one action
# =============================================================

def backup_alpha(b, a, alpha_vectors, pomdp_model, n_samples=25, gamma=0.98):
    S = len(state_list)
    alpha = np.zeros(S)
    
    # -----------------------------
    # Immediate expected reward
    # -----------------------------
    R_sa = np.zeros(S)
    for s in range(S):
        if a == "ALERT":
            # ALERT reward is timing-based → here approximate with 0
            R_sa[s] = 0
        else:
            R_sa[s] = pomdp_model["reward"][a]
    alpha += R_sa
    
    # -----------------------------
    # Future expected value (sampled)
    # -----------------------------
    for _ in range(n_samples):
        # Sample s from belief
        s = np.random.choice(S, p=b)
        
        # Sample transition + observation
        s_prime, obs = sample_transition_and_observation(s, a, pomdp_model)
        
        # Full Bayesian belief update from b, not from s
        T = pomdp_model["T"][a]
        b_pred = b @ T
        b_next = update_belief(b_pred, obs, pomdp_model)
        
        # Evaluate V(b_next)
        values = [b_next @ a_vec for a_vec in alpha_vectors]
        V_next = max(values)
        
        # Add discounted contribution
        alpha += (gamma * V_next) / n_samples
    
    return alpha


In [127]:
# =============================================================
# PBVI Step 4: Backup for all actions at belief b
# =============================================================

ACTIONS = list(pomdp_model["T"].keys())

def pbvi_backup_for_belief(b, alpha_vectors, pomdp_model):
    best_alpha = None
    best_val = -np.inf
    
    for a in ACTIONS:
        alpha_a = backup_alpha(b, a, alpha_vectors, pomdp_model)
        val = b @ alpha_a
        if val > best_val:
            best_val = val
            best_alpha = alpha_a
            
    return best_alpha


In [128]:
# =============================================================
# PBVI Step 5: One full PBVI iteration
# =============================================================

def pbvi_iteration(belief_points, alpha_vectors, pomdp_model):
    new_alphas = []
    for b in belief_points:
        new_alphas.append(pbvi_backup_for_belief(b, alpha_vectors, pomdp_model))
    return new_alphas


In [129]:
# =============================================================
# Run PBVI
# =============================================================

alpha_vectors = [np.zeros(len(state_list))]  # V_0

K = 8  # number of PBVI iterations

for it in range(K):
    alpha_vectors = pbvi_iteration(belief_points, alpha_vectors, pomdp_model)
    print(f"PBVI iteration {it+1}/{K} complete.")


PBVI iteration 1/8 complete.
PBVI iteration 2/8 complete.
PBVI iteration 2/8 complete.
PBVI iteration 3/8 complete.
PBVI iteration 3/8 complete.
PBVI iteration 4/8 complete.
PBVI iteration 4/8 complete.
PBVI iteration 5/8 complete.
PBVI iteration 5/8 complete.
PBVI iteration 6/8 complete.
PBVI iteration 6/8 complete.
PBVI iteration 7/8 complete.
PBVI iteration 7/8 complete.
PBVI iteration 8/8 complete.
PBVI iteration 8/8 complete.


In [130]:
# =============================================================
# PBVI Policy: Choose action maximizing α·b
# =============================================================

def pbvi_policy(b, alpha_vectors, pomdp_model):
    best_a = None
    best_val = -np.inf
    
    for a in ACTIONS:
        alpha_a = backup_alpha(b, a, alpha_vectors, pomdp_model)
        val = b @ alpha_a
        if val > best_val:
            best_val = val
            best_a = a

    return best_a


In [132]:
def run_cycle_pbvi(cycle_df, pomdp_model, alpha_vectors):

    S = len(state_list)
    first_phase = cycle_df["phase"].iloc[0]

    b = np.zeros(S)
    b[state_index[(first_phase,0)]] = 0.5
    b[state_index[(first_phase,1)]] = 0.5

    alerted_day = None
    queries = 0
    reward = 0

    true_onset_days = cycle_df.loc[cycle_df["true_onset"] == 1, "day"]
    if len(true_onset_days) == 0:
        return None
    true_onset_day = pd.to_datetime(true_onset_days.iloc[0])

    for _, row in cycle_df.iterrows():
        a = pbvi_policy(b, alpha_vectors, pomdp_model)

        if a == "QUERY":
            queries += 1

        if a != "ALERT":
            reward += pomdp_model["reward"][a]
        else:
            delta = (row["day"] - true_onset_day).days
            reward += pomdp_model["reward"]["ALERT"](delta)
            if alerted_day is None:
                alerted_day = row["day"]

        # Predict & update belief
        T = pomdp_model["T"][a]
        b_pred = predict_belief(b, T)

        obs = {
            "biometrics": row["biometrics"],
            "symptom": None if pd.isna(row["risk_obs"]) else int(row["risk_obs"])
        }

        b = update_belief(b_pred, obs, pomdp_model)

    delta = None if alerted_day is None else (alerted_day - true_onset_day).days

    return {
        "cycle_id": cycle_df["cycle_id"].iloc[0],
        "alert_day": alerted_day,
        "delta": delta,
        "queries": queries,
        "reward": reward
    }


In [133]:
results_complete_pbvi = []
for cid, cdf in test_complete_df_prep.groupby("cycle_id"):
    out = run_cycle_pbvi(cdf, pomdp_model, alpha_vectors)
    if out is not None:
        results_complete_pbvi.append(out)

pd.DataFrame(results_complete_pbvi)


,cycle_id,alert_day,delta,queries,reward
0,1,None,None,0,0
1,3,None,None,0,0
2,6,None,None,0,0
3,8,None,None,0,0
4,10,None,None,0,0
5,11,None,None,0,0
6,13,None,None,0,0
